In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# --- Configuration (edit here) ---
TRANSLATIONS_FOLDER = Path(r"d:\Users\Desktop\Thesis\ThesisCode\translations")

# Column mappings per dataset type
COLUMN_MAP = {
    "chartqa":     {"eng": "query",    "arb": "arabic_query"},
    "mathvision":  {"eng": "question", "arb": "arabic_question"},
}

# Length ratio thresholds
MIN_LENGTH_RATIO = 0.5
MAX_LENGTH_RATIO = 2.5

# --- Auto-discover CSV files ---
all_csv_files = sorted(TRANSLATIONS_FOLDER.glob("*.csv"))
print(f"[DEBUG] Found {len(all_csv_files)} CSV files in {TRANSLATIONS_FOLDER}")
for f in all_csv_files:
    print(f"  - {f.name}")

# --- Process each file ---
for csv_path in all_csv_files:
    fname = csv_path.name.lower()

    # Detect dataset type from filename
    if fname.startswith("chart"):
        dataset_type = "chartqa"
    elif fname.startswith("mathvision"):
        dataset_type = "mathvision"
    else:
        print(f"\n[SKIP] {csv_path.name} — not a chartqa or mathvision file.")
        continue

    cols = COLUMN_MAP[dataset_type]
    eng_col = cols["eng"]
    arb_col = cols["arb"]

    print(f"\n{'='*70}")
    print(f"FILE: {csv_path.name}")
    print(f"Type: {dataset_type}  |  English col: {eng_col}  |  Arabic col: {arb_col}")
    print(f"{'='*70}")

    # Load
    try:
        df = pd.read_csv(csv_path, encoding="utf-8")
        print(f"[DEBUG] Loaded {len(df)} rows.  Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"[ERROR] Failed to read {csv_path.name}: {e}")
        continue

    if eng_col not in df.columns or arb_col not in df.columns:
        print(f"[ERROR] Required columns missing. Need '{eng_col}' and '{arb_col}'.")
        continue

    # Clean
    df[eng_col] = df[eng_col].fillna("").astype(str).str.strip()
    df[arb_col] = df[arb_col].fillna("").astype(str).str.strip()

    # ---------------------------------------------------------
    # 1. Length Ratio Check (Hallucination Detection)
    # ---------------------------------------------------------
    df["eng_len"] = df[eng_col].str.split().str.len()
    df["ar_len"]  = df[arb_col].str.split().str.len()
    df["length_ratio"] = np.where(
        df["eng_len"] == 0, 0, df["ar_len"] / df["eng_len"]
    )
    df["pass_length"] = (
        (df["length_ratio"] >= MIN_LENGTH_RATIO) &
        (df["length_ratio"] <= MAX_LENGTH_RATIO)
    )

    # ---------------------------------------------------------
    # 2. Question Word Mapping
    # ---------------------------------------------------------
    eng_lower = df[eng_col].str.lower()

    is_how_many = eng_lower.str.startswith("how many")
    has_kam     = df[arb_col].str.contains("\u0643\u0645")

    is_what = eng_lower.str.startswith("what is") | eng_lower.str.startswith("what are")
    has_ma  = df[arb_col].str.contains("\u0645\u0627|\u0645\u0627\u0630\u0627")

    is_where = eng_lower.str.startswith("where")
    has_ayna = df[arb_col].str.contains("\u0623\u064a\u0646")

    is_when  = eng_lower.str.startswith("when")
    has_mata = df[arb_col].str.contains("\u0645\u062a\u0649")

    df["pass_q_word"] = (
        (~is_how_many | has_kam) &
        (~is_what     | has_ma)  &
        (~is_where    | has_ayna) &
        (~is_when     | has_mata)
    )

    # ---------------------------------------------------------
    # 3. Punctuation Parity
    # ---------------------------------------------------------
    eng_is_q = df[eng_col].str.endswith("?")
    ar_is_q  = df[arb_col].str.endswith("\u061f")
    df["pass_punctuation"] = (eng_is_q == ar_is_q)

    # ---------------------------------------------------------
    # Compile Results
    # ---------------------------------------------------------
    total = len(df)
    p_len  = df["pass_length"].sum()
    p_qw   = df["pass_q_word"].sum()
    p_punc = df["pass_punctuation"].sum()

    df["perfect_pass"] = df["pass_length"] & df["pass_q_word"] & df["pass_punctuation"]
    p_all = df["perfect_pass"].sum()

    print(f"\n=== Results for {csv_path.name} ===")
    print(f"Total rows: {total:,}\n")
    print(f"1. Length Ratio Pass (no hallucinations) : {p_len:>6,} ({p_len/total*100:.1f}%)")
    print(f"2. Question Word Accuracy Pass           : {p_qw:>6,} ({p_qw/total*100:.1f}%)")
    print(f"3. Punctuation Parity Pass               : {p_punc:>6,} ({p_punc/total*100:.1f}%)")
    print(f"\n\U0001F3C6 Perfect Pass (all 3 checks)          : {p_all:>6,} ({p_all/total*100:.1f}%)")

    # Clean up temp columns so they don't pile up across iterations
    df.drop(columns=["eng_len", "ar_len", "length_ratio",
                     "pass_length", "pass_q_word", "pass_punctuation",
                     "perfect_pass"], inplace=True)

print("\n" + "="*70)
print("ALL FILES PROCESSED.")
print("="*70)


[DEBUG] Found 6 CSV files in d:\Users\Desktop\Thesis\ThesisCode\translations
  - chart_aya-expansetranslation.csv
  - chart_gemma4translation.csv
  - chart_qwen3translation.csv
  - mathvision_aya-expansetranslation.csv
  - mathvision_gemma4translation.csv
  - mathvision_qwen3translation.csv

FILE: chart_aya-expansetranslation.csv
Type: chartqa  |  English col: query  |  Arabic col: arabic_query
[DEBUG] Loaded 32719 rows.  Columns: ['id', 'imgname', 'query', 'arabic_query', 'ground_truth', 'arabic_ground_truth', 'type', 'translation_time_sec']

=== Results for chart_aya-expansetranslation.csv ===
Total rows: 32,719

1. Length Ratio Pass (no hallucinations) : 32,664 (99.8%)
2. Question Word Accuracy Pass           : 32,369 (98.9%)
3. Punctuation Parity Pass               : 31,998 (97.8%)

🏆 Perfect Pass (all 3 checks)          : 31,724 (97.0%)

FILE: chart_gemma4translation.csv
Type: chartqa  |  English col: query  |  Arabic col: arabic_query
[DEBUG] Loaded 32719 rows.  Columns: ['id', '